<a href="https://colab.research.google.com/github/imdeepanshugpt/agentic-practices/blob/main/Langchain_Continue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CineBot: a movie ticket booking assistant

In [ ]:
Structured Output, Tools & Agents

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
!pip install langchain langchain-openai langchain-community langgraph python-dotenv langchain-mcp-adapters langchain-chroma chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2

In [ ]:
from langchain.chat_models import init_chat_model
model = init_chat_model('openai:gpt-5-mini')
model.invoke('Hi')
print("Cinebot's Brain is connected")

Cinebot's Brain is connected


# Structured Output

In [ ]:
booking_requests = [
    "Hi, I'd like 2 tickets for Interstellar at the 7pm show tonight, name is Priya.",
    "can u book me a seat for the 9:30 showing of dune part two? im rohan",
    "URGENT - need to CANCEL my booking for Oppenheimer, confirmation was under Aisha",
]


In [ ]:
for msg in booking_requests:
    r = model.invoke(f"Extract the customer's name, movie, and what they want (book or cancel) from: {msg}")
    print(r.content)
    print("---")


Name: Priya
Movie: Interstellar
Action: Book (2 tickets for the 7pm show tonight)
---
{
  "name": "Rohan",
  "movie": "Dune Part Two",
  "action": "book"
}
---
{
  "customer_name": "Aisha",
  "movie": "Oppenheimer",
  "request": "cancel booking"
}
---


### with_structured_output()

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class BookingRequest(BaseModel):
    customer_name: str = Field(description="The customer's name")
    movie_title: str = Field(description="The movie they want to see")
    action: Literal["book", "cancel"] = Field(description="Whether this is a new booking or a cancellation")
    ticket_count: int = Field(description="How many tickets, default 1 if not mentioned", default=1)


In [ ]:
print("Schema is defined")

Schema is defined


In [ ]:
structured_model = model.with_structured_output(BookingRequest)

In [ ]:
for msg in booking_requests:
    r = structured_model.invoke(f"Extract b booking request from: {msg}")
    print(r)
    print(f" --> action type : {type(r.action)}, value : {r.action}")
    print("---")


customer_name='Priya' movie_title='Interstellar' action='book' ticket_count=2
 --> action type : <class 'str'>, value : book
---
customer_name='Rohan' movie_title='Dune Part Two' action='book' ticket_count=1
 --> action type : <class 'str'>, value : book
---
customer_name='Aisha' movie_title='Oppenheimer' action='cancel' ticket_count=1
 --> action type : <class 'str'>, value : cancel
---


In [ ]:
r

BookingRequest(customer_name='Aisha', movie_title='Oppenheimer', action='cancel', ticket_count=1)

# Tool Strategy & Provider Strategy

Two different mechanisms achieve the same guarantee. `ProviderStrategy` uses the model
provider's own native structured-output feature (fast, but only works where supported).
`ToolStrategy` fakes it via a synthetic tool call (works almost everywhere, slightly slower).

In [ ]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

In [ ]:
provider_strategy_model = model.with_structured_output(BookingRequest, strategy=ProviderStrategy(BookingRequest))

In [ ]:
model.profile

{'name': 'GPT-5 Mini',
 'release_date': '2025-08-07',
 'last_updated': '2025-08-07',
 'open_weights': False,
 'max_input_tokens': 272000,
 'max_output_tokens': 128000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': False,
 'image_url_inputs': True,
 'pdf_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True,
 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']}

In [ ]:
model_3 = init_chat_model("openai:gpt-3.5-turbo")


In [ ]:
model_3.profile

{'name': 'GPT-3.5-turbo',
 'release_date': '2023-03-01',
 'last_updated': '2023-11-06',
 'open_weights': False,
 'max_input_tokens': 16385,
 'max_output_tokens': 4096,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': False,
 'structured_output': False,
 'attachment': False,
 'temperature': True,
 'image_url_inputs': False,
 'pdf_inputs': False,
 'pdf_tool_message': False,
 'image_tool_message': False,
 'tool_choice': True,
 'tool_call_streaming': True}

In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent


class Answer(BaseModel):
    summary: str
    confidence: float


agent = create_agent(model="openai:gpt-3.5-turbo", response_format=ToolStrategy(Answer) # Will fail.
result = agent.invoke({"messages": [{"role": "user", "content": "Summarize AI trends"}]})
result["structured_response"]  # Answer(summary=..., confidence=...)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!"
    )
)

agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})